# Real-data pipeline — rolling VaR/ES on Fama–French

Walks through `alg:real-data` (§9) end-to-end on the synthetic FF3 panel.
We:

1. Load a Fama–French panel (offline-synthetic by default; same API for live).
2. Fit Hill / POT-GPD tail indices.
3. Compute pairwise $\chi$ and $\eta$ dependence diagnostics.
4. Run the rolling VaR/ES forecast at 99% and 99.5%.
5. Backtest with Kupiec, Christoffersen, dynamic-quantile, and
   Acerbi–Szekely.

The offline-synthetic surrogate produces a deterministic heavy-tailed
panel with the same shape and column names as the public Kenneth-French
files.

In [ ]:
import numpy as np
import pandas as pd

## 1. Load the panel

In [ ]:
from factortail.real_data import load_fama_french

panel = load_fama_french(name='FF3_daily', offline=True, n_synthetic=2000, rng_seed=0)
print(f'panel name    : {panel.name}')
print(f'source        : {panel.source}')
print(f'rows          : {panel.n_obs}')
print(f'date range    : {panel.start_date.date()} -> {panel.end_date.date()}')
print(f'missing rate  : {panel.missing_rate:.4f}')
print(f'checksum      : {panel.checksum}')
panel.data.head()

## 2. Marginal tail indices

Hill and POT-GPD on the right tail of each factor.

In [ ]:
from factortail.diagnostics import hill_estimator, pot_gpd_estimator

rows = []
for col in panel.data.columns.drop('RF', errors='ignore'):
    pos = panel.data[col].to_numpy()
    pos = pos[pos > 0]
    k = int(0.1 * pos.size)
    hill = hill_estimator(pos, k=k)
    pot = pot_gpd_estimator(pos, k=k)
    rows.append({
        'factor': col,
        'hill_alpha': hill['alpha_hat'],
        'pot_alpha':  pot['alpha_hat'],
        'k':          k,
    })
pd.DataFrame(rows)

## 3. Pairwise dependence

$\chi(u)$, $\bar\chi(u)$, and Ledford–Tawn $\eta$ for every factor
pair.

In [ ]:
from factortail.diagnostics import pairwise_dependence_table

df_factors = panel.data.drop(columns=['RF'], errors='ignore')
table = pairwise_dependence_table(
    df_factors.to_numpy(),
    threshold_u=0.95,
    eta_k=50,
    column_names=list(df_factors.columns),
)
table

## 4. Rolling VaR / ES

Algorithm `alg:real-data` with a 400-day window.

In [ ]:
from factortail.real_data import RollingVaRConfig, run_rolling_var_es

portfolio = 'Mkt-RF'
series = panel.data[portfolio]
factors = panel.data.drop(columns=[portfolio, 'RF'], errors='ignore')
rc = RollingVaRConfig(window=400, levels=(0.99, 0.995), n_inner=3000, seed=0)
df_var = run_rolling_var_es(series, factors, portfolio=portfolio, config=rc)
df_var.head()

## 5. Backtests

Kupiec / Christoffersen / dynamic-quantile / Acerbi–Szekely Z2.

In [ ]:
from factortail.real_data.backtests import (
    acerbi_szekely_es,
    christoffersen_test,
    dq_test,
    kupiec_test,
)

rows = []
for level in rc.levels:
    sub = df_var[df_var['level'] == level].dropna()
    hits = sub['hit'].to_numpy()
    kup = kupiec_test(hits, level=level)
    chris = christoffersen_test(hits, level=level)
    dq = dq_test(hits, level=level)
    es = acerbi_szekely_es(sub['loss'].to_numpy(), sub['var'].to_numpy(), sub['es'].to_numpy(), level=level)
    rows.append({
        'level':           level,
        'observed_hits':   kup['observed_hits'],
        'expected_hits':   round(kup['expected_hits'], 1),
        'kupiec_p':        round(kup['p_value'], 4),
        'christoffersen_p':round(chris['p_value'], 4),
        'dq_p':            round(dq['p_value'], 4),
        'es_z2':           round(es['statistic'], 4),
    })
pd.DataFrame(rows)

## What you've just reproduced

This is essentially the §9 pipeline that produces `F16_var_es_dashboard`
and `T_var_es_backtest_placeholder`. Every CSV under `results/` is
schema-validated and stamped with `run_id` / `config_hash` /
`git_hash` / `code_version` / `run_timestamp` — see
[Reproducibility](../reproducibility.md) and the App. G replacement
contract enforced by `factortail validate-run`.

For CRSP licensed extensions (P7), see
[Data and DGPs](../datasets.md).